In [ ]:
# ================================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ================================================================
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
import joblib

# ================================================================
# 2. CARGA DE DATOS DE ENTRENAMIENTO
# (Reemplaza por tu archivo real)
# ================================================================
df = pd.read_csv("/kaggle/input/udea-ai-4-eng-20252-pruebas-saber-pro-colombia/train.csv")

# Separar variables
X = df.drop("objetivo", axis=1)   # <-- cambia "objetivo" por tu variable real
y = df["objetivo"]

# Identificar columnas categóricas y numéricas
columnas_categoricas = X.select_dtypes(include=["object"]).columns
columnas_numericas = X.select_dtypes(exclude=["object"]).columns

# ================================================================
# 3. PREPROCESAMIENTO CON OneHotEncoder
# ================================================================
preprocesador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), columnas_categoricas),
        ("num", "passthrough", columnas_numericas)
    ]
)

# ================================================================
# 4. CREAR PIPELINE COMPLETO (preprocesador + modelo)
# ================================================================
modelo = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("rf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        n_jobs=-1
    ))
])

# ================================================================
# 5. ENTRENAR MODELO COMPLETO
# ================================================================
modelo.fit(X, y)

# Guardar pipeline entero
joblib.dump(modelo, "modelo_rf_pipeline.pkl")
print("Modelo guardado como modelo_rf_pipeline.pkl")

# ================================================================
# 6. PREDICCIÓN EN KAGGLE CON CSV NUEVO
# ================================================================
df_test = pd.read_csv("/kaggle/input/udea-ai-4-eng-20252-pruebas-saber-pro-colombia/train.csv")

# Predecir directamente (ya incluye preprocesamiento)
pred = modelo.predict(df_test)

# Crear CSV de salida
salida = pd.DataFrame({
    "id": df_test.index,
    "prediccion": pred
})

salida.to_csv("submission.csv", index=False)
print("Archivo submission.csv generado correctamente.")
